In [27]:
from transformers import AutoTokenizer
TOKENIZER_PATH = "/mnt/cfs-rd/models/qwen3/Qwen3-4B-Thinking-2507/"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH, trust_remote_code=True)


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [28]:
PROMPT_CRITIC_TEMPLATE = """# ROLE
You are a Socratic Logic Probe. Your goal is to guide the user (who provided the `CoT`) to self-correct their mistakes. You are talking directly to them.

# INPUT DATA
## CONTEXT
{context_fmt_str}

## STATEMENT
{statement}

## CORRECT_ANSWER (HIDDEN TRUTH)
{correct_answer}

## CoT (User's Chain of Thought)
{result}

# CRITICAL RULES (NON-NEGOTIABLE)
1. **VOICE & TONE:** In your final output, speak directly to the user. ALWAYS use "**you**" and "**your**" to refer to the user's reasoning.
2. **NO SPOILERS:** NEVER reveal the `CORRECT_ANSWER` or the final classification (True/False).
3. **LOCATE THE ERROR:** Scan the `CoT` specifically looking for the `## Step n` markers. Find the *first* step where **the user's** logic deviates from the `CONTEXT` or misinterprets the `STATEMENT`.
4. **QUESTION THE PREMISE:** Formulate a question that targets **the user's** specific misunderstanding in that Step.
5. **MANDATORY REDIRECTION:** You MUST conclude by instructing the user to discard **their** reasoning from that step onwards (e.g., "Please re-examine logic...").

# WORKFLOW
1. **Compare:** Match each `## Step n` in the `CoT` against the `CONTEXT`.
2. **Identify:** Find the specific Step (e.g., "Step 2") where **the user** started hallucinating or making a logic error.
3. **Draft Question:** Create a leading question based ONLY on the `CONTEXT` evidence relevant to that Step.
   - *Drafting Mental Check:* The question must ask "In Step X, **you** stated..." (Address the user as 'you' in the final draft).
4. **Output:** Structure your response strictly as defined below, maintaining the second-person perspective throughout.

# RESPONSE FORMAT
<analysis>
[Briefly explain to the user why the specific Step **you found** is the root cause of **their** error. Address them directly. E.g., "You misinterpreted the relationship between X and Y here..."]
</analysis>

<guidance>
[Your Socratic Question pointing to the evidence **the user** ignored, phrased directly to them.]
**Please re-examine the logic in ## Step [n] and re-evaluate your conclusion from there.**
</guidance>"""
print(len(tokenizer.encode(PROMPT_CRITIC_TEMPLATE, add_special_tokens=False)))

530


In [4]:
"""
Theoretical FLOPs Calculation for GRPO vs. SOPO (ICML 2026 Rebuttal)
Based on Kaplan et al. (2020) Scaling Laws:
- Forward Pass (Generation/Inference): ~ 2 * N * D FLOPs
- Backward Pass (Gradient Update):     ~ 6 * N * D FLOPs
Where N = Active Parameters, D = Tokens processed (including Prefill).
"""

# ==========================================
# 1. Model & Training Hyperparameters
# ==========================================
epochs = 4
n_samples = 10000
n_traj_per_sample = 8
total_trajectories = epochs * n_samples * n_traj_per_sample  # T = 320,000

# Active Parameters (N)
param_student = 4 * 10**9
# CRITICAL: Our 80B Teacher is an MoE model with only 3B ACTIVE parameters per token.
param_teacher = 3 * 10**9

# ==========================================
# 2. Empirical Log Data (Average Tokens)
# ==========================================
# Student Prompt
prompt_student = 959.73  # includes length of system prompt and context/sample

# Teacher Prompt Breakdown
sys_prompt_teacher = 530.0
avg_prompt_teacher_context = 646.48
avg_prompt_teacher_statement = 165.55
avg_prompt_teacher_answer = 73.50

# Sequence lengths from empirical training logs
avg_gen_len_sopo = 2529.41
avg_gen_len_grpo = 1998.11
avg_gen_len_teacher_per_turn = 211.18

# Teacher Invocation Expectation
# Avg turns = 1.57 means the teacher is implicitly invoked 0.57 times per trajectory.
avg_turn_sopo = 1.57
expected_teacher_calls = avg_turn_sopo - 1.0

# Dynamic Sampling Ratio (Crucial for Efficiency)
# Socratic feedback drives the policy to high confidence rapidly.
# We filter out zero-variance groups (where advantages are all 0 or 1).
# Thus, the heavy backward pass is only executed on 48.02% of the trajectories.
dynamic_sampling_ratio = 0.4802
# dynamic_sampling_ratio = 1

# ==========================================
# 3. Token Breakdown (Expectation per Trajectory)
# ==========================================
# Expected generated tokens by Teacher
expected_teacher_gen = avg_gen_len_teacher_per_turn * expected_teacher_calls

# Expected generated tokens by Student
avg_gen_len_sopo_student = avg_gen_len_sopo - expected_teacher_gen

# Teacher's Context (Prefix): When invoked, it reads the static prompt AND the student's previous turn.
avg_student_gen_per_turn = avg_gen_len_sopo_student / avg_turn_sopo
total_tokens_per_teacher_call = (
    sys_prompt_teacher
    + avg_prompt_teacher_context
    + avg_prompt_teacher_statement
    + avg_prompt_teacher_answer
    + avg_student_gen_per_turn
    + avg_gen_len_teacher_per_turn
)
expected_teacher_total_tokens = total_tokens_per_teacher_call * expected_teacher_calls

# ==========================================
# 4. Strict FLOPs Calculation (Forward vs. Backward)
# ==========================================

# --- Baseline: Standard GRPO ---
# No reference KL used. 100% of T participates in both Forward (2ND) and Backward (6ND).
# Multiplier = 2ND + 6ND = 8ND
flops_grpo = (
    8 * param_student * (avg_gen_len_grpo + prompt_student) * total_trajectories
)

# --- Ours: SOPO Framework ---
# Phase A: Forward Generation (100% of T MUST be generated to compute rewards)
flops_sopo_student_gen = (
    2 * param_student * (avg_gen_len_sopo_student + prompt_student) * total_trajectories
)
flops_sopo_teacher_gen = (
    2 * param_teacher * expected_teacher_total_tokens * total_trajectories
)

# Phase B: Backward Update (Applied ONLY to the dynamically sampled subset, ρ = 48.02%)
# The student backpropagates through its entire history (avg_gen_len_sopo + prompt_student).
flops_sopo_student_update = (
    6
    * param_student
    * (avg_gen_len_sopo + prompt_student)
    * total_trajectories
    * dynamic_sampling_ratio
)

# Total SOPO Epoch FLOPs
flops_sopo_total = (
    flops_sopo_student_gen + flops_sopo_teacher_gen + flops_sopo_student_update
)

# ==========================================
# 5. Output Formatting (in ExaFLOPs)
# ==========================================
EFLOP = 10**18

print(f"--- Theoretical Total Training Compute (Full Run) ---")
print(f"GRPO (Baseline): {flops_grpo / EFLOP:.2f} EFLOPs")
print(f"SOPO (Total):    {flops_sopo_total / EFLOP:.2f} EFLOPs")
print(f"  ├─ Forward: Student (100%): {flops_sopo_student_gen / EFLOP:.2f} EFLOPs")
print(f"  ├─ Forward: Teacher (100%): {flops_sopo_teacher_gen / EFLOP:.2f} EFLOPs")
print(f"  └─ Backward: Student (48%): {flops_sopo_student_update / EFLOP:.2f} EFLOPs")

overhead_percent = ((flops_sopo_total - flops_grpo) / flops_grpo) * 100

if overhead_percent > 0:
    print(f"\n=> Conclusion 1 (Full Epoch Cost): SOPO introduces a strictly marginal overhead of +{overhead_percent:.1f}% compared to GRPO.")
else:
    print(f"\n=> Conclusion 1 (Full Epoch Cost): SOPO is {abs(overhead_percent):.1f}% CHEAPER than standard GRPO.")

# ==========================================
# 6. Iso-Performance Calculation (Accelerated Convergence)
# ==========================================
# Dense Socratic feedback allows SOPO to reach baseline's peak performance in ~50% steps.
step_w_same_perf = 0.50
flops_sopo_iso = flops_sopo_total * step_w_same_perf

iso_saving = ((flops_grpo - flops_sopo_iso) / flops_grpo) * 100
print(f"\n--- Iso-Performance Compute ---")
print(f"To reach the target baseline accuracy:")
print(f"SOPO FLOPs Required: {flops_sopo_iso / EFLOP:.2f} EFLOPs")
print(
    f"=> Conclusion 2 (Net Savings): SOPO reduces total Iso-Performance compute by {iso_saving:.1f}%!"
)

--- Theoretical Total Training Compute (Full Run) ---
GRPO (Baseline): 30.29 EFLOPs
SOPO (Total):    24.95 EFLOPs
  ├─ Forward: Student (100%): 8.62 EFLOPs
  ├─ Forward: Teacher (100%): 3.46 EFLOPs
  └─ Backward: Student (48%): 12.87 EFLOPs

=> Conclusion 1 (Full Epoch Cost): SOPO is 17.6% CHEAPER than standard GRPO.

--- Iso-Performance Compute ---
To reach the target baseline accuracy:
SOPO FLOPs Required: 12.48 EFLOPs
=> Conclusion 2 (Net Savings): SOPO reduces total Iso-Performance compute by 58.8%!


In [11]:
avg_student_gen_per_turn

1534.4187261146494

In [8]:
avg_gen_len_sopo_student+prompt_student

3368.7673999999997

In [9]:
avg_gen_len_grpo+prompt_student

2957.84

In [5]:
print(sys_prompt_teacher
    + avg_prompt_teacher_context
    + avg_prompt_teacher_statement
    + avg_prompt_teacher_answer
    + avg_student_gen_per_turn)

2735.4058659217876


## REBUTTAL V2

In [3]:
"""
Theoretical FLOPs Calculation for GRPO vs. SOPO (ICML 2026 Rebuttal)
Based on Kaplan et al. (2020) Scaling Laws:
- Forward Pass (Generation/Inference): ~ 2 * N * D FLOPs
- Backward Pass (Gradient Update):     ~ 6 * N * D FLOPs
Where N = Active Parameters, D = Tokens processed (including Prefill).
"""

# ==========================================
# 1. Model & Training Hyperparameters
# ==========================================
epochs = 4
n_samples = 10000
n_traj_per_sample = 8
total_trajectories = epochs * n_samples * n_traj_per_sample  # T = 320,000

# Active Parameters (N)
param_student = 4 * 10**9
# CRITICAL: Our 80B Teacher is an MoE model with only 3B ACTIVE parameters per token.
param_teacher = 3 * 10**9

# ==========================================
# 2. Empirical Log Data (Average Tokens)
# ==========================================
# Student Prompt
prompt_student = 959.73  # includes length of system prompt and context/sample

# Teacher Prompt Breakdown
sys_prompt_teacher = 530.0
avg_prompt_teacher_context = 646.48
avg_prompt_teacher_statement = 165.55
avg_prompt_teacher_answer = 73.50

# Sequence lengths from empirical training logs
avg_gen_len_sopo = 2529.41
avg_gen_len_grpo = 2700.1
avg_gen_len_teacher_per_turn = 211.18

# Teacher Invocation Expectation
# Avg turns = 1.57 means the teacher is implicitly invoked 0.57 times per trajectory.
avg_turn_sopo = 1.57
expected_teacher_calls = avg_turn_sopo - 1.0

# Dynamic Sampling Ratio (Crucial for Efficiency)
# Socratic feedback drives the policy to high confidence rapidly.
# We filter out zero-variance groups (where advantages are all 0 or 1).
# Thus, the heavy backward pass is only executed on 48.02% of the trajectories.
dynamic_sampling_ratio = 0.4802
dynamic_sampling_ratio_grpo = 0.4130
# dynamic_sampling_ratio = 1

# ==========================================
# 3. Token Breakdown (Expectation per Trajectory)
# ==========================================
# Expected generated tokens by Teacher
expected_teacher_gen = avg_gen_len_teacher_per_turn * expected_teacher_calls

# Expected generated tokens by Student
avg_gen_len_sopo_student = avg_gen_len_sopo - expected_teacher_gen

# Teacher's Context (Prefix): When invoked, it reads the static prompt AND the student's previous turn.
avg_student_gen_per_turn = avg_gen_len_sopo_student / avg_turn_sopo
total_tokens_per_teacher_call = (
    sys_prompt_teacher
    + avg_prompt_teacher_context
    + avg_prompt_teacher_statement
    + avg_prompt_teacher_answer
    + avg_student_gen_per_turn
    + avg_gen_len_teacher_per_turn
)
expected_teacher_total_tokens = total_tokens_per_teacher_call * expected_teacher_calls

# ==========================================
# 4. Strict FLOPs Calculation (Forward vs. Backward)
# ==========================================

# --- Baseline: Standard GRPO ---
# No reference KL used. 100% of T participates in both Forward (2ND) and Backward (6ND).
# Multiplier = 2ND + 6ND = 8ND

flops_grpo_forward = 2 * param_student * (avg_gen_len_grpo + prompt_student) * total_trajectories
flops_grpo_backward = 6 * param_student * (avg_gen_len_grpo + prompt_student) * total_trajectories * dynamic_sampling_ratio_grpo

flops_grpo = flops_grpo_forward + flops_grpo_backward

# --- Ours: SOPO Framework ---
# Phase A: Forward Generation (100% of T MUST be generated to compute rewards)
flops_sopo_student_gen = (
    2 * param_student * (avg_gen_len_sopo_student + prompt_student) * total_trajectories
)
flops_sopo_teacher_gen = (
    2 * param_teacher * expected_teacher_total_tokens * total_trajectories
)

# Phase B: Backward Update (Applied ONLY to the dynamically sampled subset, ρ = 48.02%)
# The student backpropagates through its entire history (avg_gen_len_sopo + prompt_student).
flops_sopo_student_update = (
    6
    * param_student
    * (avg_gen_len_sopo + prompt_student)
    * total_trajectories
    * dynamic_sampling_ratio
)

# Total SOPO Epoch FLOPs
flops_sopo_total = (
    flops_sopo_student_gen + flops_sopo_teacher_gen + flops_sopo_student_update
)

# ==========================================
# 5. Output Formatting (in ExaFLOPs)
# ==========================================
EFLOP = 10**18

print(f"--- Theoretical Total Training Compute (Full Run) ---")
print(f"GRPO (Baseline): {flops_grpo / EFLOP:.2f} EFLOPs")
print(f"  ├─ Forward: {flops_grpo_forward / EFLOP:.2f} EFLOPs")
print(f"  └─ Backward: {flops_grpo_backward / EFLOP:.2f} EFLOPs")
print(f"SOPO (Total):    {flops_sopo_total / EFLOP:.2f} EFLOPs")
print(f"  ├─ Forward: Student (100%): {flops_sopo_student_gen / EFLOP:.2f} EFLOPs")
print(f"  ├─ Forward: Teacher (100%): {flops_sopo_teacher_gen / EFLOP:.2f} EFLOPs")
print(f"  └─ Backward: Student (48%): {flops_sopo_student_update / EFLOP:.2f} EFLOPs")

overhead_percent = ((flops_sopo_total - flops_grpo) / flops_grpo) * 100

if overhead_percent > 0:
    print(f"\n=> Conclusion 1 (Full Epoch Cost): SOPO introduces a strictly marginal overhead of +{overhead_percent:.1f}% compared to GRPO.")
else:
    print(f"\n=> Conclusion 1 (Full Epoch Cost): SOPO is {abs(overhead_percent):.1f}% CHEAPER than standard GRPO.")

# ==========================================
# 6. Iso-Performance Calculation (Accelerated Convergence)
# ==========================================
# Dense Socratic feedback allows SOPO to reach baseline's peak performance in ~50% steps.
step_w_same_perf = 0.50
flops_sopo_iso = flops_sopo_total * step_w_same_perf

iso_saving = ((flops_grpo - flops_sopo_iso) / flops_grpo) * 100
print(f"\n--- Iso-Performance Compute ---")
print(f"To reach the target baseline accuracy:")
print(f"SOPO FLOPs Required: {flops_sopo_iso / EFLOP:.2f} EFLOPs")
print(
    f"=> Conclusion 2 (Net Savings): SOPO reduces total Iso-Performance compute by {iso_saving:.1f}%!"
)

--- Theoretical Total Training Compute (Full Run) ---
GRPO (Baseline): 20.98 EFLOPs
  ├─ Forward: 9.37 EFLOPs
  └─ Backward: 11.61 EFLOPs
SOPO (Total):    24.95 EFLOPs
  ├─ Forward: Student (100%): 8.62 EFLOPs
  ├─ Forward: Teacher (100%): 3.46 EFLOPs
  └─ Backward: Student (48%): 12.87 EFLOPs

=> Conclusion 1 (Full Epoch Cost): SOPO introduces a strictly marginal overhead of +18.9% compared to GRPO.

--- Iso-Performance Compute ---
To reach the target baseline accuracy:
SOPO FLOPs Required: 12.48 EFLOPs
=> Conclusion 2 (Net Savings): SOPO reduces total Iso-Performance compute by 40.5%!


In [14]:
### SOPO wo DS
"""
Theoretical FLOPs Calculation for GRPO vs. SOPO (ICML 2026 Rebuttal)
Based on Kaplan et al. (2020) Scaling Laws:
- Forward Pass (Generation/Inference): ~ 2 * N * D FLOPs
- Backward Pass (Gradient Update):     ~ 6 * N * D FLOPs
Where N = Active Parameters, D = Tokens processed (including Prefill).
"""

# ==========================================
# 1. Model & Training Hyperparameters
# ==========================================
epochs = 4
n_samples = 10000
n_traj_per_sample = 8
total_trajectories = epochs * n_samples * n_traj_per_sample  # T = 320,000

# Active Parameters (N)
param_student = 4 * 10**9
# CRITICAL: Our 80B Teacher is an MoE model with only 3B ACTIVE parameters per token.
param_teacher = 3 * 10**9

# ==========================================
# 2. Empirical Log Data (Average Tokens)
# ==========================================
# Student Prompt
prompt_student = 959.73  # includes length of system prompt and context/sample

# Teacher Prompt Breakdown
sys_prompt_teacher = 530.0
avg_prompt_teacher_context = 646.48
avg_prompt_teacher_statement = 165.55
avg_prompt_teacher_answer = 73.50

# Sequence lengths from empirical training logs
avg_gen_len_sopo = 2245
avg_gen_len_grpo = 1998.1
avg_gen_len_teacher_per_turn = 172.80

# Teacher Invocation Expectation
# Avg turns = 1.57 means the teacher is implicitly invoked 0.57 times per trajectory.
avg_turn_sopo = 1.52
expected_teacher_calls = avg_turn_sopo - 1.0

# Dynamic Sampling Ratio (Crucial for Efficiency)
# Socratic feedback drives the policy to high confidence rapidly.
# We filter out zero-variance groups (where advantages are all 0 or 1).
# Thus, the heavy backward pass is only executed on 48.02% of the trajectories.
# dynamic_sampling_ratio = 0.4802
# dynamic_sampling_ratio_grpo = 0.4130
dynamic_sampling_ratio = dynamic_sampling_ratio_grpo = 1
# dynamic_sampling_ratio = 1

# ==========================================
# 3. Token Breakdown (Expectation per Trajectory)
# ==========================================
# Expected generated tokens by Teacher
expected_teacher_gen = avg_gen_len_teacher_per_turn * expected_teacher_calls

# Expected generated tokens by Student
avg_gen_len_sopo_student = avg_gen_len_sopo - expected_teacher_gen

# Teacher's Context (Prefix): When invoked, it reads the static prompt AND the student's previous turn.
avg_student_gen_per_turn = avg_gen_len_sopo_student / avg_turn_sopo
total_tokens_per_teacher_call = (
    sys_prompt_teacher
    + avg_prompt_teacher_context
    + avg_prompt_teacher_statement
    + avg_prompt_teacher_answer
    + avg_student_gen_per_turn
    + avg_gen_len_teacher_per_turn
)
expected_teacher_total_tokens = total_tokens_per_teacher_call * expected_teacher_calls

# ==========================================
# 4. Strict FLOPs Calculation (Forward vs. Backward)
# ==========================================

# --- Baseline: Standard GRPO ---
# No reference KL used. 100% of T participates in both Forward (2ND) and Backward (6ND).
# Multiplier = 2ND + 6ND = 8ND

flops_grpo_forward = 2 * param_student * (avg_gen_len_grpo + prompt_student) * total_trajectories
flops_grpo_backward = 6 * param_student * (avg_gen_len_grpo + prompt_student) * total_trajectories * dynamic_sampling_ratio_grpo

flops_grpo = flops_grpo_forward + flops_grpo_backward

# --- Ours: SOPO Framework ---
# Phase A: Forward Generation (100% of T MUST be generated to compute rewards)
flops_sopo_student_gen = (
    2 * param_student * (avg_gen_len_sopo_student + prompt_student) * total_trajectories
)
flops_sopo_teacher_gen = (
    2 * param_teacher * expected_teacher_total_tokens * total_trajectories
)

# Phase B: Backward Update (Applied ONLY to the dynamically sampled subset, ρ = 48.02%)
# The student backpropagates through its entire history (avg_gen_len_sopo + prompt_student).
flops_sopo_student_update = (
    6
    * param_student
    * (avg_gen_len_sopo + prompt_student)
    * total_trajectories
    * dynamic_sampling_ratio
)

# Total SOPO Epoch FLOPs
flops_sopo_total = (
    flops_sopo_student_gen + flops_sopo_teacher_gen + flops_sopo_student_update
)

# ==========================================
# 5. Output Formatting (in ExaFLOPs)
# ==========================================
EFLOP = 10**18

print(f"--- Theoretical Total Training Compute (Full Run) ---")
print(f"GRPO (Baseline): {flops_grpo / EFLOP:.2f} EFLOPs")
print(f"  ├─ Forward: {flops_grpo_forward / EFLOP:.2f} EFLOPs")
print(f"  └─ Backward: {flops_grpo_backward / EFLOP:.2f} EFLOPs")
print(f"SOPO (Total):    {flops_sopo_total / EFLOP:.2f} EFLOPs")
print(f"  ├─ Forward: Student (100%): {flops_sopo_student_gen / EFLOP:.2f} EFLOPs")
print(f"  ├─ Forward: Teacher (100%): {flops_sopo_teacher_gen / EFLOP:.2f} EFLOPs")
print(f"  └─ Backward: Student (48%): {flops_sopo_student_update / EFLOP:.2f} EFLOPs")

overhead_percent = ((flops_sopo_total - flops_grpo) / flops_grpo) * 100

if overhead_percent > 0:
    print(f"\n=> Conclusion 1 (Full Epoch Cost): SOPO introduces a strictly marginal overhead of +{overhead_percent:.1f}% compared to GRPO.")
else:
    print(f"\n=> Conclusion 1 (Full Epoch Cost): SOPO is {abs(overhead_percent):.1f}% CHEAPER than standard GRPO.")

# ==========================================
# 6. Iso-Performance Calculation (Accelerated Convergence)
# ==========================================
# Dense Socratic feedback allows SOPO to reach baseline's peak performance in ~50% steps.
step_w_same_perf = 0.50
flops_sopo_iso = flops_sopo_total * step_w_same_perf

iso_saving = ((flops_grpo - flops_sopo_iso) / flops_grpo) * 100
print(f"\n--- Iso-Performance Compute ---")
print(f"To reach the target baseline accuracy:")
print(f"SOPO FLOPs Required: {flops_sopo_iso / EFLOP:.2f} EFLOPs")
print(
    f"=> Conclusion 2 (Net Savings): SOPO reduces total Iso-Performance compute by {iso_saving:.1f}%!"
)

--- Theoretical Total Training Compute (Full Run) ---
GRPO (Baseline): 30.29 EFLOPs
  ├─ Forward: 7.57 EFLOPs
  └─ Backward: 22.72 EFLOPs
SOPO (Total):    35.59 EFLOPs
  ├─ Forward: Student (100%): 7.97 EFLOPs
  ├─ Forward: Teacher (100%): 3.00 EFLOPs
  └─ Backward: Student (48%): 24.61 EFLOPs

=> Conclusion 1 (Full Epoch Cost): SOPO introduces a strictly marginal overhead of +17.5% compared to GRPO.

--- Iso-Performance Compute ---
To reach the target baseline accuracy:
SOPO FLOPs Required: 17.79 EFLOPs
=> Conclusion 2 (Net Savings): SOPO reduces total Iso-Performance compute by 41.3%!


In [15]:
avg_student_gen_per_turn

1417.857894736842